# CG2 v7 Gradio app on Google Colab (free GPU)

The v7 Gradio app (`v7_gradio.zip`): upload a pottery photo → Grounding DINO + SAM 2.1 + CLIP cut out the ceramic object →
ColorThief glaze colours → the colour-match recipe model (text–chemistry space + CIEDE2000 colour + surface + cone) →
recipe cards (Regular / Diversified, CSV download). Here it runs on Colab's GPU and you get a public **gradio.live** link.

**How to use**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. Cell 2 shows **Choose Files**: pick `v7_gradio.zip` (on the PC: `D:\NUS study\CG2\GC2-Ensemble-Model-Pottery\v7_gradio.zip`).
4. Wait about 5–10 minutes. Cell 5 prints the link.
5. Stop: **Runtime → Disconnect and delete runtime**.

The app's own `app.py` runs unchanged; the models (Grounding DINO, SAM 2.1, CLIP, MiniLM, about 2 GB) are downloaded in
cell 4 so the first **Detect pottery** is quick. The link works while this notebook runs (about 90 minutes without
activity or about 12 hours at most on free Colab; the link itself expires after 72 hours).

In [ ]:
# 1 · Check the GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "no GPU")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again"
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# 2 · Upload v7_gradio.zip and unpack it (the macOS __MACOSX folder is left out)
import pathlib, zipfile
from google.colab import files
ZIP = pathlib.Path("/content/v7_gradio.zip")
if not ZIP.exists():
    up = files.upload()                                   # choose v7_gradio.zip
    ZIP.write_bytes(up[next(iter(up))])
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content/v7", members=[m for m in z.namelist() if not m.startswith("__MACOSX/")])
APP = next(str(p.parent) for p in pathlib.Path("/content/v7").rglob("app.py"))
print("app folder:", APP)
print(sorted(p.name for p in pathlib.Path(APP).iterdir()))

In [ ]:
# 3 · Install the app's packages (requirements-hf.txt) + Gradio 5.50. Colab's own torch / torchvision stay.
import re
req = [l.strip() for l in open(f"{APP}/requirements-hf.txt") if l.strip() and not l.lstrip().startswith("#")]
req = [l for l in req if not re.match(r"(torch|torchvision)\b", l)]
open("/content/req_v7.txt", "w").write("\n".join(req + ["gradio==5.50.0"]) + "\n")
!pip install -q -r /content/req_v7.txt

In [ ]:
# 4 · Download the models now into the app's model cache (<folder above the app>/model_cache), so the first click is quick
import os, pathlib, subprocess, sys
CACHE = str(pathlib.Path(APP).parent / "model_cache" / "huggingface")
code = """
from huggingface_hub import snapshot_download
for repo, pat in [("IDEA-Research/grounding-dino-base", ["*.json", "*.txt", "model.safetensors"]),
                  ("facebook/sam2.1-hiera-large", ["*.json", "model.safetensors"]),
                  ("sentence-transformers/all-MiniLM-L6-v2", ["*.json", "*.txt", "model.safetensors", "1_Pooling/*"]),
                  ("laion/CLIP-ViT-B-32-laion2B-s34B-b79K", ["open_clip_*"])]:
    print(repo, snapshot_download(repo, allow_patterns=pat), flush=True)
"""
subprocess.run([sys.executable, "-c", code], check=True, env=dict(os.environ, HF_HOME=CACHE, HF_HUB_CACHE=CACHE + "/hub"))

In [ ]:
# 5 · Start the app with a public link (Gradio share link, *.gradio.live)
import os, re, subprocess, sys, time
LOG = "/content/v7_gradio.log"
env = dict(os.environ, GRADIO_SHARE="True", PYTHONUNBUFFERED="1")
proc = subprocess.Popen([sys.executable, "app.py"], cwd=APP, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
url, t0 = None, time.time()
while url is None:
    time.sleep(3)
    txt = open(LOG, errors="ignore").read()
    m = re.search(r"https://[a-z0-9-]+\.gradio\.live", txt)
    if m and True:
        url = m.group(0)
    elif proc.poll() is not None:
        print(txt[-4000:])
        raise SystemExit("The app stopped: see the end of its log above.")
    elif time.time() - t0 > 1800:
        print(txt[-4000:])
        raise SystemExit("No link after 30 minutes: see the log above.")
print(f"\nREADY after {time.time() - t0:.0f} s. Open (or share) this link:\n\n    {url}\n")

In [ ]:
# 6 · Keep the app running and show its log. The link works while this cell runs and the Colab session is open.
# Stop: press the ■ stop button of this cell, then Runtime > Disconnect and delete runtime.
import time
pos = 0
try:
    while proc.poll() is None:
        with open(LOG, errors="ignore") as f:
            f.seek(pos)
            new = f.read()
            pos = f.tell()
        if new:
            print(new, end="")
        time.sleep(5)
    print("\nThe app stopped (exit code", proc.returncode, ")")
except KeyboardInterrupt:
    proc.terminate()
    print("Stopped.")